# Customer Segmentation using K-Means Clustering
This notebook builds a production-quality customer segmentation pipeline. It selects key customer behavior features, standardizes them, determines the optimal number of clusters, trains a K-Means model, profiles and visualizes the segments, and exports the final model and data.

## 0. Load Dataset and Setup
We import libraries and load the feature-engineered dataset.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import joblib
import os

# Set default plotly template
pio.templates.default = "plotly_white"

# Load engineered features
df = pd.read_csv("../dataset/processed/customer_features.csv")
print(f"Dataset shape: {df.shape}")

## 1. Feature Selection and Rationale
We select the key behavior and demographic features for clustering and document their selection rationale.

In [ ]:
# Define clustering features
clustering_features = [
    'Income', 
    'Age', 
    'Total_Spending', 
    'Total_Purchases', 
    'Average_Spending_Per_Purchase', 
    'Customer_Tenure'
]

X = df[clustering_features]
X.head()

### Feature Selection Rationale:
1. **`Income`**: Core indicator of customer purchasing power. Helps distinguish between budget-conscious and affluent customer segments.
2. **`Age`**: Traditional demographic variable to identify generational preferences and life stages (e.g. young professionals vs retired seniors).
3. **`Total_Spending`**: Summarizes the customer's total value (LTV proxy) to the business across all product categories.
4. **`Total_Purchases`**: Reflects customer transaction frequency. Differentiates between low-frequency high-ticket buyers and high-frequency low-ticket buyers.
5. **`Average_Spending_Per_Purchase`**: Cart value proxy. Indicates transaction efficiency and average basket size.
6. **`Customer_Tenure`**: Customer relationship duration. Measures brand loyalty and registration cohort age.

## 2. Standardization
We standardize the selected features using `StandardScaler` to bring them onto the same scale. This is crucial for K-Means as it relies on Euclidean distance.

In [ ]:
# Instantiate and fit the scaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Create scaled dataframe for preview
X_scaled_df = pd.DataFrame(X_scaled, columns=clustering_features)
X_scaled_df.head()

## 3. Determine Optimal Number of Clusters
We evaluate cluster sizes from $K=2$ to $K=10$ using the **Elbow Method** (inertia) and **Silhouette Score** to choose the best number of segments.

In [ ]:
inertia_scores = []
silhouette_scores = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia_scores.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, kmeans.labels_))

# Plot Elbow Curve (Inertia)
fig_elbow = px.line(x=list(K_range), y=inertia_scores, markers=True,
                    title='Elbow Method for Optimal K',
                    labels={'x': 'Number of Clusters (K)', 'y': 'Inertia (Within-Cluster Sum of Squares)'},
                    color_discrete_sequence=['#4B7BEC'])
fig_elbow.show()

# Plot Silhouette Scores
fig_sil = px.line(x=list(K_range), y=silhouette_scores, markers=True,
                  title='Silhouette Score for Optimal K',
                  labels={'x': 'Number of Clusters (K)', 'y': 'Silhouette Coefficient'},
                  color_discrete_sequence=['#20BF6B'])
fig_sil.show()

### Optimal K Decision:
* **Silhouette Score** peaks at $K=2$ (0.376) and remains stable around $K=3$ (0.250) and $K=4$ (0.231).
* **Elbow Method** shows a gradual bend starting at $K=3$ and flattening noticeably at $K=4$.
* From a business perspective, **$K=4$ is chosen** as it splits the customer base into four highly distinct, balanced, and actionable segments (VIPs, Mature Value Shoppers, Frugal Loyalists, and New Frugal Shoppers).

## 4. Train Final Model and Assign Labels
We train the K-Means model with $K=4$ and assign the cluster labels to our customer records.

In [ ]:
# Train final KMeans model
optimal_k = 4
kmeans_model = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df['Cluster'] = kmeans_model.fit_predict(X_scaled)

print("Cluster distributions:")
print(df['Cluster'].value_counts().sort_index())

## 5. Segment Profiling
Let's look at the customer counts and averages of demographic, spending, and engagement attributes for each cluster.

In [ ]:
# Profile segments with core aggregation metrics
profile = df.groupby('Cluster').agg({
    'ID': 'count',
    'Income': 'mean',
    'Total_Spending': 'mean',
    'Age': 'mean',
    'Total_Purchases': 'mean',
    'Average_Spending_Per_Purchase': 'mean',
    'Customer_Tenure': 'mean',
    'Response': 'mean'
}).rename(columns={'ID': 'Customer_Count', 'Response': 'Campaign_Response_Rate'}).reset_index()

# Format profile columns for output
profile['Campaign_Response_Rate'] = (profile['Campaign_Response_Rate'] * 100).round(2).astype(str) + '%'
profile = profile.round(2)
profile

### Cluster Profiles Overview:

#### **Cluster 0: Mature Value Shoppers (Stable Core)**
* **Customer Count**: 623 (~28% of base)
* **Financials**: Mid-to-high income (~$65k) with solid overall spending (~$868).
* **Demographics**: Older cohort (avg age: 54 years).
* **Behavior**: High purchase frequency (18.25 purchases) with mid-range cart value ($47.58). Moderate marketing campaign interest (11.88% Response).
* **Characteristics**: Stable, mature middle-class shoppers who buy regularly and value reliability.

#### **Cluster 1: High-Value VIPs (Elite Spenders)**
* **Customer Count**: 438 (~20% of base)
* **Financials**: Highest income (~$76k) and extremely high spending (~$1,492).
* **Demographics**: Younger-to-mid career cohort (avg age: 39.5 years).
* **Behavior**: Highest purchase frequency (19.82 purchases) and massive cart values ($82.58). Very high campaign acceptance rate (32.19% Response).
* **Characteristics**: Tech-savvy, high-earning, premium consumers. They drive disproportionate revenue and respond heavily to high-end campaigns.

#### **Cluster 2: Frugal Loyalists (Disengaged Bargain Hunters)**
* **Customer Count**: 541 (~24% of base)
* **Financials**: Low income (~$33.8k) and low spending (~$169).
* **Demographics**: Younger-to-mid career cohort (avg age: 40.9 years).
* **Behavior**: Moderate purchase frequency (7.42 purchases) but very long brand relationship (longest average Customer_Tenure: 526.6 days). High campaign responsiveness (16.08% Response).
* **Characteristics**: Budget-conscious but loyal customers. They respond strongly to discounts, sales, and promo campaigns.

#### **Cluster 3: Unengaged Starters (New Frugals)**
* **Customer Count**: 634 (~28% of base)
* **Financials**: Low-to-mid income (~$37.9k) and lowest spending (~$108).
* **Demographics**: Middle-aged cohort (avg age: 43.6 years).
* **Behavior**: Shortest relationship (avg Customer_Tenure: 172.8 days), low purchase counts (6.29), and very low marketing engagement (5.05% Response).
* **Characteristics**: Newly acquired customers with low transaction rates. Represent a cold segment requiring nurturing to avoid churn.

## 6. Visualization
We visualize the segments using PCA dimensional reduction, size breakdowns, spending comparisons, and a comparative radar chart.

In [ ]:
# 1. PCA 2D Cluster Plot
pca = PCA(n_components=2, random_state=42)
pca_comps = pca.fit_transform(X_scaled)
df['PC1'] = pca_comps[:, 0]
df['PC2'] = pca_comps[:, 1]

fig_pca = px.scatter(df, x='PC1', y='PC2', color=df['Cluster'].astype(str),
                     title='2D PCA Projection of Customer Segments',
                     labels={'color': 'Segment / Cluster'},
                     color_discrete_sequence=px.colors.qualitative.Bold,
                     opacity=0.7)
fig_pca.show()

# 2. Cluster Size Bar Chart
size_counts = df['Cluster'].value_counts().reset_index()
size_counts.columns = ['Cluster', 'Count']
fig_size = px.bar(size_counts, x='Cluster', y='Count', title='Customer Count by Segment',
                  labels={'Cluster': 'Segment/Cluster', 'Count': 'Number of Customers'},
                  color=size_counts['Cluster'].astype(str), color_discrete_sequence=px.colors.qualitative.Bold)
fig_size.show()

# 3. Cluster Spending Boxplot
fig_spend_box = px.box(df, x='Cluster', y='Total_Spending', title='Spending Distribution by Segment',
                       color=df['Cluster'].astype(str), color_discrete_sequence=px.colors.qualitative.Bold)
fig_spend_box.show()

# 4. Cluster Income Boxplot
fig_inc_box = px.box(df, x='Cluster', y='Income', title='Income Distribution by Segment',
                     color=df['Cluster'].astype(str), color_discrete_sequence=px.colors.qualitative.Bold)
fig_inc_box.show()

# 5. Radar Chart Comparing Standardized Centers
scaled_features_df = pd.DataFrame(X_scaled, columns=clustering_features)
scaled_features_df['Cluster'] = df['Cluster']
cluster_means = scaled_features_df.groupby('Cluster').mean().reset_index()

melted_means = pd.melt(cluster_means, id_vars=['Cluster'], value_vars=clustering_features,
                       var_name='Feature', value_name='Standardized_Mean')

fig_radar = go.Figure()
colors_radar = ['#0080FF', '#FF007F', '#00CC66', '#FF9900']

for c in sorted(melted_means['Cluster'].unique()):
    c_data = melted_means[melted_means['Cluster'] == c]
    fig_radar.add_trace(go.Scatterpolar(
        r=c_data['Standardized_Mean'].tolist() + [c_data['Standardized_Mean'].iloc[0]],
        theta=c_data['Feature'].tolist() + [c_data['Feature'].iloc[0]],
        fill='toself',
        name=f'Cluster {c}',
        line=dict(color=colors_radar[c])
    ))

fig_radar.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[-2, 2.5])),
    title='Radar Chart of Segment Profiles (Standardized Means)',
    showlegend=True
)
fig_radar.show()

## 7. Business Recommendations
Based on the characteristics of each cluster, we define specific marketing actions to maximize retention, cart values, and marketing ROI.

### **Cluster 0 (Mature Value Shoppers - Stable Core)**
* **Action**: Retain with loyalty clubs and regular product updates.
* **Strategies**: Offer targeted catalog campaigns showcasing new wines or gourmet meats. Run subscription packages (e.g., "Wine of the Month") to lock in continuous purchasing. Avoid aggressive pricing cuts since they are not price-sensitive, but focus on reliability and quality.

### **Cluster 1 (High-Value VIPs - Elite Spenders)**
* **Action**: Leverage high purchasing power and campaign interest to cross-sell premium additions.
* **Strategies**: Direct invitation to premium programs. Roll out personalized early-access promotions. Since they have a 32% response rate, design customized premium email marketing offering high-end product pairings (e.g., cellared wines and organic cuts). Avoid generic mass discounts, which could dilute brand equity.

### **Cluster 2 (Frugal Loyalists - Disengaged Bargain Hunters)**
* **Action**: Drive transaction volume using discounts and coupons.
* **Strategies**: Deliver high-frequency discount communications, flash sale alerts, and coupon bundles. Use bulk volume incentives (e.g., "Buy 2 get 1 free"). Focus on low-cost categories like fruits and gold products where margins are flexible. Keep them engaged to maintain their long tenure.

### **Cluster 3 (Unengaged Starters - New Frugals)**
* **Action**: Nurture to prevent early-stage churn.
* **Strategies**: Deploy onboarding welcome campaigns offering low-barrier first-purchase discounts. Provide engaging guides or tutorials on using the online portal to increase Web visits. Monitor their inactivity closely and trigger automated win-back emails if they fail to make a second purchase within 60 days.

## 8. Export Outputs
We save the labeled customer dataset and serialize our fitted models (`scaler` and `kmeans`) to disk.

In [ ]:
# Export segments dataset
segments_export_path = "../dataset/processed/customer_segments.csv"
df.to_csv(segments_export_path, index=False)
print(f"Labeled customer dataset exported to: {segments_export_path}")

# Create models folder if not exists
os.makedirs("../models", exist_ok=True)

# Serialize models
joblib.dump(scaler, "../models/scaler.pkl")
joblib.dump(kmeans_model, "../models/kmeans_model.pkl")
print("StandardScaler and KMeans models serialized successfully under '../models/' folder.")